# Efficient Task Allocation for Multiple Load Carrying Robots in Warehouse Environment

**Authors:** Sachin Verma, Raj Krishnan Vijayaraj and Iris Iris Uwizeyimana<br>
**Course:** ECE1724H: Bio-inspired Algorithms for Smart Mobility - Fall 2022<br>
**Instructor:** Dr. Alaa Khamis<br>
**Department:** Edward S. Rogers Sr. Department of Electrical & Computer Engineering, University of Toronto

## Introduction

With the rapid evolution of automation technologies and growing confidence in autonomous systems, self- driving robots are gradually becoming popular in warehouse operations and management. Collaborative work has been proven to reduce processing time and quality of operations and the adoption of autonomous robots, specifically in large scale warehouse environments, is expected to deliver high efficiencies and increase the throughput of the overall logistics ecosystem. However, there are existing challenges that prevent the complete autonomy of Multi-Robot Systems (MRS) in the warehouse environment.  

This work primarily focuses on tackling the *Multi-Robot Task Allocation (MRTA)* problem using a *Reinforced Learning (RL)* based method and an *Discrete Adaptive Particle Swarm Optimisation (DAPSO)* technique. The task allocation problem needs to tackle multiple objectives; 
1. Minimizing the power consumption of the available robots 
2. Minimising the overall time taken to complete all tasks
3. Achieving good load balance across the available robots 

Our approach also incorporates optimized path planning for each robot with a designated task.

## Installing Libraries
The following libraries are used in the project:
1. tqdm -> Progress meter for iteration
2. networkx -> To generate the network layout of the warehouse
3. ffmpeg -> To generate animation using bar-chart-race
4. bar-chart-race -> To generate animated bar chart races in Python with matplotlib
5. stable-baselines3 -> For installing RL dependencies

In [1]:
!pip install tqdm
!pip install networkx
!pip install ffmpeg-python
!pip install bar-chart-race
!pip install stable-baselines3

## Dataset

### Robot dataset
This is synthetically generated using the following code:
The parameters that define battery are: 
1. bdf -> battery discharge factor
2. location -> starting location
3. init_battery -> initial battery level

In [2]:
def create_robot_dataset(num_bots, init_battery=100):
  bots = []
  for j in range(num_bots): 
    # battery discharge factor is assumed as:
    # bdf = 100/(load*time_to_discharge)
    bdf = 100/(20*1000)
    # location = random.randint(1001,1002) #(0,j)
    location = 1000
    bots.append([bdf,location,init_battery])
  return bots

### Load dataset
This is synthetically generated using the following code:

In [3]:
def create_load_dataset(num_loads, pickup_loc, grid_size):
  loads = []
  for i in range(num_loads):
    #pick a random value between [5,25] for the load's weight
    weight = random.randint(5, 25)
    #compute the dropoff_locations
    dropoff_loc = random.randint(1,grid_size)
    loads.append([weight, pickup_loc, dropoff_loc])  
  return loads

### Layout Generator

This is synthetically generated using the following code:

In [ ]:
class LayoutNetworkX:
    def __init__(self,
                num_rack=4,
                units_per_rack=5,
                ):
        self.num_rack = num_rack
        self.units_per_rack = units_per_rack
        self.num_aisle = self.num_rack + 1
        self.aisle_width = 9
        self.rack_width = 5
        self.unit_width = 4
        self.first_rack_x = 10 # location of bottom-left rack
        self.first_rack_y = 5
        self.staging_hub_x = 0  #auto-computed in node_generator based on aisle and rack parameters
        self.staging_hub_y = 0  
        self.staging_node_id = 1000 #staging region node series
        self.unit_location_x = []
        self.unit_location_y = []
        self.unit_node_id = []
        self.total_unit_node = 0
        self.unit_node_data_dict = {}
        self.path_location_x = []
        self.path_location_y = []
        self.path_node_id = []
        self.total_path_node = 0
        self.path_node_data_dict = {}
        self.stage_location_x = []
        self.stage_location_y = []
        self.stage_node_id = []
        self.stage_node_data_dict = {}
        self.all_node_data_dict = {}
        self.unit_edge = []
        self.aisle_edge = []
        self.fasttrack_edge = []
        self.edge_attribute = {}
        self.Graph = nx.empty_graph(0)
        self.route_lookup = {}
        
        self.speed_unit = 0.75  #robot speed in edges connecting aisle and units
        self.speed_aisle = 1.25 #robot speed in aisle edges
        self.speed_fasttrack = 2 #robot speed in Fast-track edges
        
    
    def node_generator(self):
        
        unit_location_x = []
        unit_location_y = []
        unit_node_id = []
        id_temp = 1

        # Generating Nodes at Unit Storage Locations 
        for i in range(self.num_rack):
            for k in range(2):
                for j in range(self.units_per_rack):
                    if k == 0:
                        unit_node_id.append(str(id_temp))
                        unit_location_x.append(self.first_rack_x + (i * (self.rack_width + self.aisle_width)))
                        unit_location_y.append(self.first_rack_y +(self.unit_width/2) + (j * self.unit_width ))
                        id_temp+=1
                    if k == 1:
                        unit_node_id.append(str(id_temp))
                        unit_location_x.append(self.first_rack_x + (i * (self.rack_width + self.aisle_width) + self.rack_width))
                        unit_location_y.append(self.first_rack_y +(self.unit_width/2) + (j * self.unit_width ))
                        id_temp+=1

        unit_node_location = list(zip(unit_location_x, unit_location_y))
        unit_node_data = list(zip(unit_node_id, unit_location_x, unit_location_y))
        unit_node_data_dict = dict(zip(unit_node_id, unit_node_location))

        
        self.total_unit_node = len(unit_node_id)
        self.unit_location_x = unit_location_x
        self.unit_location_y = unit_location_y
        self.unit_node_id = unit_node_id
        self.unit_node_data_dict = unit_node_data_dict
        
        
        # Generating Path Nodes
        path_location_x = []
        path_location_y = []
        path_node_id = []

        for i in range(self.num_rack+1):
            for j in range(self.units_per_rack):
                if j == 0:
                    path_node_id.append(str(id_temp))
                    path_location_x.append(self.first_rack_x - self.aisle_width/2 + (i * (self.rack_width + self.aisle_width)))
                    path_location_y.append(self.first_rack_y - self.aisle_width/2)
                    id_temp+=1

                path_node_id.append(str(id_temp))
                path_location_x.append(self.first_rack_x - self.aisle_width/2 + (i * (self.rack_width + self.aisle_width)))
                path_location_y.append(self.first_rack_y +(self.unit_width/2) + j * self.unit_width)
                id_temp+=1

                if j == self.units_per_rack-1:
                    path_node_id.append(str(id_temp))
                    path_location_x.append(self.first_rack_x - self.aisle_width/2 + (i * (self.rack_width + self.aisle_width)))
                    path_location_y.append(self.first_rack_y + self.units_per_rack*self.unit_width + self.aisle_width/2)
                    id_temp+=1


        
        path_node_location = list(zip(path_location_x, path_location_y))
        path_node_data = list(zip(path_node_id, path_location_x, path_location_y))
        path_node_data_dict = dict(zip(path_node_id, path_node_location))
        
        self.total_path_node = len(path_node_id)
        self.path_location_x = path_location_x 
        self.path_location_y = path_location_y
        self.path_node_id = path_node_id
        self.path_node_data_dict = path_node_data_dict
        
        # Generating Nodes for Staging Area
        x_loc = self.first_rack_x - self.aisle_width/2 - (self.rack_width + self.aisle_width)
        self.stage_location_x = [x_loc,
                                 x_loc,
                                 x_loc
                                ]
        
        y_loc_max = self.first_rack_y + self.units_per_rack*self.unit_width + self.aisle_width/2
        y_loc_min = self.first_rack_y - self.aisle_width/2
        y_loc_centre = (y_loc_max + y_loc_min)/2
        self.stage_location_y = [y_loc_centre,
                                 y_loc_min,
                                 y_loc_max
                                ]
        
        self.stage_node_id = [str(self.staging_node_id),
                              str(self.staging_node_id+1),
                              str(self.staging_node_id+2)
                             ]
        
        stage_node_location = list(zip(self.stage_location_x, self.stage_location_y))
        self.stage_node_data_dict = dict(zip(self.stage_node_id, stage_node_location))
        
        # | operator de
        #self.all_node_data_dict = self.unit_node_data_dict | self.path_node_data_dict | self.stage_node_data_dict
        self.all_node_data_dict = {**self.unit_node_data_dict, **self.path_node_data_dict}
        self.all_node_data_dict = {**self.all_node_data_dict, **self.stage_node_data_dict}

        # self.all_node_data_dict = self.unit_node_data_dict.copy()
        # self.all_node_data_dict.update(self.path_node_data_dict)
        # self.all_node_data_dict.update(self.all_node_data_dict)
    
    def find_distance(self, node1, node2):
        p1_arr = np.asarray(self.all_node_data_dict[node1])
        p2_arr = np.asarray(self.all_node_data_dict[node2])
        distance = np.linalg.norm(p1_arr-p2_arr)
        return distance
        
    def edge_generator(self):
        
        # Connecting Aisle Nodes to Aisle Nodes, and Unit Nodes to Aisle Nodes
        unit_edge = []
        aisle_edge = []
        edge_attribute = {}
        counter1 = 0 #unit number
        counter2 = 0 #unit_node_index
        counter3 = 0 #aisle number

        # Rules to find the edge pair is based on num_rack and units_per_rack
        # Note: Separate Rules are defined for the boundary nodes to 
        # generate the desired warehouse layout

        for n in self.path_node_id:
            
            if counter1 == 0:
                aisle_edge.append((n, str(int(n)+1)))
                edge_attribute[(n, str(int(n)+1))] = {'distance':self.find_distance(str(n), str(int(n)+1)),
                                                      'speed':self.speed_aisle,
                                                      'type':'aisle'}
                counter1 += 1

                if counter3 == 1:
                    counter2 += 1
                    continue

                if counter3 > 1:
                    counter2 += self.units_per_rack + 1
                    continue

                continue

            if counter1 == self.units_per_rack:
                aisle_edge.append((n, str(int(n)+1)))
                edge_attribute[(n, str(int(n)+1))] = {'distance':self.find_distance(str(n), str(int(n)+1)),
                                                      'speed':self.speed_aisle, 
                                                      'type':'aisle'}
                
                if counter3 > 0 and counter3 < self.num_aisle-1:
                    unit_edge.append((n, self.unit_node_id[counter2 + self.units_per_rack]))
                    edge_attribute[(n, self.unit_node_id[counter2 + self.units_per_rack])] = {'distance':self.find_distance(str(n), str(self.unit_node_id[counter2 + self.units_per_rack])),
                                                                                              'speed':self.speed_unit,
                                                                                              'type':'unit'}
                    unit_edge.append((n, self.unit_node_id[counter2]))
                    edge_attribute[(n, self.unit_node_id[counter2])] = {'distance':self.find_distance(str(n), str(self.unit_node_id[counter2])),
                                                                        'speed':self.speed_unit,
                                                                        'type':'unit'}

                if counter3 == self.num_aisle-1:
                    unit_edge.append((n, self.unit_node_id[counter2]))
                    edge_attribute[(n, self.unit_node_id[counter2])] = {'distance':self.find_distance(str(n), str(self.unit_node_id[counter2])),
                                                                        'speed':self.speed_unit,
                                                                        'type':'unit'}

                if counter3 == 0:
                    unit_edge.append((n, self.unit_node_id[counter2]))
                    edge_attribute[(n, self.unit_node_id[counter2])] = {'distance':self.find_distance(str(n), str(self.unit_node_id[counter2])),
                                                                        'speed':self.speed_unit,
                                                                        'type':'unit'}

                counter1 += 1
                continue

            if counter1 > self.units_per_rack:
                counter1 = 0
                counter3 += 1
                continue

            if counter3 > 0 and counter3 < self.num_aisle-1:
                aisle_edge.append((n, str(int(n)+1)))
                edge_attribute[(n, str(int(n)+1))] = {'distance':self.find_distance(str(n), str(int(n)+1)),
                                                      'speed':self.speed_aisle,
                                                      'type':'aisle'}
                unit_edge.append((n, self.unit_node_id[counter2 + self.units_per_rack]))
                edge_attribute[(n, self.unit_node_id[counter2 + self.units_per_rack])] = {'distance':self.find_distance(str(n), str(self.unit_node_id[counter2 + self.units_per_rack])),
                                                                                          'speed':self.speed_unit,
                                                                                          'type':'unit'}
                unit_edge.append((n, self.unit_node_id[counter2]))
                edge_attribute[(n, self.unit_node_id[counter2])] = {'distance':self.find_distance(str(n), str(self.unit_node_id[counter2])),
                                                                    'speed':self.speed_unit,
                                                                    'type':'unit'}
                counter2 += 1
                counter1 += 1
                continue

            if counter3 == self.num_aisle-1:
                aisle_edge.append((n, str(int(n)+1)))
                edge_attribute[(n, str(int(n)+1))] = {'distance':self.find_distance(str(n), str(int(n)+1)),
                                                      'speed':self.speed_aisle,
                                                      'type':'aisle'}
                
                unit_edge.append((n, self.unit_node_id[counter2]))
                edge_attribute[(n, self.unit_node_id[counter2])] = {'distance':self.find_distance(str(n), str(self.unit_node_id[counter2])),
                                                                    'speed':self.speed_unit,
                                                                    'type':'unit'}
                counter2 += 1
                counter1 += 1
                continue

            if counter3 == 0:
                aisle_edge.append((n, str(int(n)+1)))
                edge_attribute[(n, str(int(n)+1))] = {'distance':self.find_distance(str(n), str(int(n)+1)),
                                                      'speed':self.speed_aisle,
                                                      'type':'aisle'}
                
                unit_edge.append((n, self.unit_node_id[counter2]))
                edge_attribute[(n, self.unit_node_id[counter2])] = {'distance':self.find_distance(str(n), str(self.unit_node_id[counter2])),
                                                                    'speed':self.speed_unit,
                                                                    'type':'unit'}
                counter2 += 1
                counter1 += 1
                continue


        # Connecting Fast-Track Nodes to Fast-Track Nodes
        fasttrack_edge = []
        for i,n1 in enumerate(self.path_node_id[0:-1:self.units_per_rack+2]):
            if i < self.num_aisle-1:
                fasttrack_edge.append((n1,str(int(n1) + self.units_per_rack+2)))
                edge_attribute[(n1,str(int(n1) + self.units_per_rack+2))] = {'distance':self.find_distance(str(n1),str(int(n1) + self.units_per_rack+2)),
                                                                             'speed':self.speed_fasttrack,
                                                                             'type':'fast_track'}

        for i,n1 in enumerate(self.path_node_id[self.units_per_rack+1:-1:self.units_per_rack+2]):
            if i < self.num_aisle-1:
                fasttrack_edge.append((n1,str(int(n1) + self.units_per_rack+2)))
                edge_attribute[(n1,str(int(n1) + self.units_per_rack+2))] = {'distance':self.find_distance(str(n1),str(int(n1) + self.units_per_rack+2)),
                                                                             'speed':self.speed_fasttrack,
                                                                             'type':'fast_track'}

        self.unit_edge = unit_edge
        self.aisle_edge = aisle_edge
        self.fasttrack_edge = fasttrack_edge
        self.edge_attribute = edge_attribute
        
        
        # Connecting Staging Area Node to Fast-Track Nodes
        bottom_left_node = self.num_rack*self.units_per_rack*2 + 1
        top_left_node = bottom_left_node + self.units_per_rack + 1
        fasttrack_edge.append((str(bottom_left_node), str(self.staging_node_id+1)))
        edge_attribute[(str(bottom_left_node), str(self.staging_node_id+1))] = {'distance':self.find_distance(str(bottom_left_node), str(self.staging_node_id+1)),
                                                                                'speed':self.speed_fasttrack,
                                                                                'type':'fast_track'}
        
        fasttrack_edge.append((str(self.staging_node_id+1), str(self.staging_node_id)))
        edge_attribute[(str(self.staging_node_id+1), str(self.staging_node_id))] = {'distance':self.find_distance(str(self.staging_node_id+1), str(self.staging_node_id)),
                                                                                    'speed':self.speed_fasttrack,
                                                                                    'type':'fast_track'}
        
        fasttrack_edge.append((str(self.staging_node_id), str(self.staging_node_id+2)))
        edge_attribute[(str(self.staging_node_id), str(self.staging_node_id+2))] = {'distance':self.find_distance(str(self.staging_node_id), str(self.staging_node_id+2)),
                                                                                    'speed':self.speed_fasttrack,
                                                                                    'type':'fast_track'}
        
        fasttrack_edge.append((str(self.staging_node_id+2), str(top_left_node)))
        edge_attribute[(str(self.staging_node_id+2), str(top_left_node))] = {'distance':self.find_distance(str(self.staging_node_id+2), str(top_left_node)),
                                                                             'speed':self.speed_fasttrack,
                                                                             'type':'fast_track'}
        
    def visualize_layout(self, route = None):
        fig, ax = plt.subplots(figsize=(8,8), dpi=150) 
        
        pos_unit_node = self.unit_node_data_dict
        pos_path_node = self.path_node_data_dict
        pos_stage_node = self.stage_node_data_dict
        pos_all_node = pos_unit_node | pos_path_node | pos_stage_node
        label_unit = dict(zip(self.unit_node_id,self.unit_node_id))
        label_path = dict(zip(self.path_node_id,self.path_node_id))
        label_stage = dict(zip(self.stage_node_id,self.stage_node_id))

        # Drawing unit nodes
        nx.draw_networkx_nodes(self.Graph, pos=pos_all_node, nodelist=pos_unit_node.keys(), node_color="black",node_size=250,node_shape="s")
        nx.draw_networkx_labels(self.Graph, pos=pos_all_node, labels = label_unit, font_size=10, font_color="white")
        
        # Drawing path nodes
        nx.draw_networkx_nodes(self.Graph, pos=pos_all_node, nodelist=pos_path_node.keys(), node_color="gray",node_size=100)
        nx.draw_networkx_labels(self.Graph, pos=pos_all_node, labels = label_path, font_size=8, font_color="black")
        
        # Drawing staging area nodes
        nx.draw_networkx_nodes(self.Graph, pos=pos_all_node, nodelist=pos_stage_node.keys(), node_color="orange",node_size=100)
        nx.draw_networkx_labels(self.Graph, pos=pos_all_node, labels = label_stage, font_size=8, font_color="black")
       
        # Drawing Edges
        nx.draw_networkx_edges(self.Graph, pos=pos_all_node, edgelist=self.unit_edge, width=2, edge_color="tab:red")
        nx.draw_networkx_edges(self.Graph, pos=pos_all_node, edgelist=self.aisle_edge, width=3, edge_color="tab:blue")
        nx.draw_networkx_edges(self.Graph, pos=pos_all_node, edgelist=self.fasttrack_edge, width=4, edge_color="green")
        
        if route!=None:
            route_edges = list(zip(route,route[1:]))
            # nx.draw_networkx_nodes(self.Graph, pos=pos_all_node,nodelist=route, node_color='r')
            nx.draw_networkx_edges(self.Graph, pos=pos_all_node, edgelist=route_edges,edge_color='r',width=5)
            
        plt.show()
        print("Total Unit Nodes: ", self.total_unit_node)
        print("Total Path Nodes: ", self.total_path_node)

        
    def generate_graph(self):
        self.node_generator()
        self.edge_generator()
        G = nx.Graph()
        G.add_nodes_from(self.unit_node_id)
        G.add_nodes_from(self.path_node_id)
        G.add_nodes_from(self.stage_node_id)
        G.add_edges_from(self.unit_edge)
        G.add_edges_from(self.aisle_edge)
        G.add_edges_from(self.fasttrack_edge)
        nx.set_edge_attributes(G, self.edge_attribute)
        self.Graph = G
        return self.Graph
    
    # Heuristic function for NetwrokX single_source_bellman_ford
    def time_cost(a,b,c,d):
        cost = d['distance']/d['speed']
        return cost
    
    def best_route_helper(self,node_id_src, node_id_dst):
        cost, route = nx.single_source_bellman_ford(self.Graph, str(node_id_src), str(node_id_dst), weight=self.time_cost)
        return round(cost,2), route
        
    def best_route_store(self):
        for stage_node_id in self.stage_node_id:
            for unit_node_id in self.unit_node_id:
                cost, route = self.best_route_helper(unit_node_id, stage_node_id)
                self.route_lookup.update({(unit_node_id,stage_node_id) : cost})
                self.route_lookup.update({(stage_node_id,unit_node_id) : cost})

        for stage_node_id1 in self.stage_node_id:
          for stage_node_id2 in self.stage_node_id:
            # if stage_node_id1 != stage_node_id2:
              cost, route = self.best_route_helper(unit_node_id, stage_node_id)
              self.route_lookup.update({(stage_node_id1,stage_node_id2) : cost})
              self.route_lookup.update({(stage_node_id2,stage_node_id1) : cost})



        
        
                
    def best_route(self, node_id_src, node_id_dst):
        cost = self.route_lookup[(str(node_id_src),str(node_id_dst))]
        return cost



## Problem Formulation

The objective of MRTA is to generate a tasking schedule for robots to minimize the overall operation costs in a warehouse setting.

The staging area consists of feeder belts that supply the incoming load packages to the robots. Once a task is allocated, the robot transports the load form the staging area to the designated unit in a rack. It is assumed that robots travel in  pre-assigned lanes and that the average speeds in different lanes are known.  Further, to simulate the unavailability of a robot, we discharge the robot's battery based on the assigned load's weight and the travel distance. The robots are then recharged in the charging zones before they take up the next assigned task.   

### Mathematical formulation,  
The warehouse layout is expressed as a graph $G(V, E)$ where the vertices $V \in \{V_P, V_D, V_J\}$ and edges $E \in \{E_U, E_A, E_S\}$. We consider $V_F$ as the feeder nodes, $V_U$ as the unit (load drop-off) nodes, $V_J$ as the junction nodes connecting multiple edges, $E_U$ as low-speed tracks connect aisle and units, $E_A$ as medium speed tracks along the aisle, and $E_S$ as high-speed tracks. Each edge is assigned a weight based on its track speed. 

## Reinforcement Learning Approach

### MRTA Environment

In [ ]:
class MrtaEnv(Env):
  def __init__(self, load_queue, robots, num_robots, task_window, charge_wait, num_rack, units_per_rack):
    self.network = LayoutNetworkX(num_rack, units_per_rack)
    self.graph = self.network.generate_graph()
    self.network.best_route_store() #create lookup table for all possible routes

    # Each robot has 3 attributes: 0) battery discharge factor, 
    # 1) current location, 2)initial battery
    self.robots = robots

    self.num_robots = num_robots

    # keep track of each robot's queue length and current location
    self.r_queue = np.zeros(num_robots)
    self.r_masked_iteration = np.ones(num_robots)
    self.r_masked_iteration = -1*self.r_masked_iteration

    # Track Total and Average Rewards For the episode
    self.episode_reward_sum = 0
    self.episode_reward_mean = 0
    self.tot_episodes_rew_sum = []
    self.tot_episodes_rew_mean = []
    
    self.r_locs = []
    self.r_battery = []
    for bot in robots:
      self.r_locs.append(bot[1])
      self.r_battery.append(bot[2])

    # waiting time for robot charging
    self.charge_wait = charge_wait

    # metadata that contains how many times the robot had to recharge
    self.per_robot_charge_freq = np.zeros(num_robots)

    # metadata that contains what loads got assigned to the different robots
    # initiallly empty 
    self.assigned_loads_per_robot = np.empty((num_robots), dtype=object)
    for r in range(num_robots):
      self.assigned_loads_per_robot[r] = []

    # Each load in a queue has 6 attributes
    # 0) load weight, 1) pickup location, 2) drop-off location
    # 3) task deadline, 4) arrival_time
    self.load_queue = load_queue

    self.task_window = task_window

    # keep track of the number of completed load
    # and the index to the first non-assigned task within a window size
    self.finished_loads = 0
    self.first_task_pointers = np.zeros(task_window, dtype=int)
    for i in range(task_window):
      self.first_task_pointers[i] = i

    # our action space is of size num_robots * task_window
    # action chooses which robots gets assigned which task on a give time stamp
    # action%task_window = task to be assigned (within task window)
    # action/num_robots = chosen robot
    self.action_space = Discrete(num_robots * task_window)

    # unmasked version
    self.observation_space = Box(low=0.0, high=10e5, shape=(len(load_queue), ((2*num_robots)+1)), dtype=float)
    self.current_iteration_time = 0
    self.state = np.zeros((len(load_queue), ((2*num_robots)+1)), dtype=float)
    self.update_state_table()
    self.longest_queue_length = 0 
    self.legal_actions = np.ones(num_robots * task_window, dtype=bool)

      
  def find_path(self, source, destination):
    cost = self.network.best_route(source,destination)
    return cost

# updates the state table that carries the task completion time for each robot
# the column attributes are:  [0]-> task done/notdone status
#                             [odd_idx] -> time-till_start for robot r
#                             [even_idx] -> total task completion time for robot r
                              
  def update_state_table(self):
    for l_id_ in range(len(self.state)):
      for r_idx_ in range(self.num_robots):
        robot_loc = self.r_locs[r_idx_]
        load_pickup = self.load_queue[l_id_][1]
        load_dropoff = self.load_queue[l_id_][2]
        time_pick = self.find_path(robot_loc, load_pickup)
        time_drop = self.find_path(load_pickup, load_dropoff)
        duration_time = time_pick + time_drop
        time_till_start = self.r_queue[r_idx_]
        self.state[l_id_][(r_idx_*2)+1] = time_till_start
        self.state[l_id_][(r_idx_*2)+2] = duration_time


  def update_task_window_pointer(self):
    # Note: instead of masking, make sure the task window always points to the
    # last load 
    task_window_idx = 0

    #last load index
    last_l_idx = 0

    for l_idx_ in range(len(self.load_queue)):

      if(task_window_idx == self.task_window):
        break

      #check if this load from the state table has been initialized
      if( self.state[l_idx_][0] == 1):
        continue

      self.first_task_pointers[task_window_idx] = l_idx_
      task_window_idx += 1
      last_l_idx = l_idx_

    # case when available tasks are less than the task window
    # reaching end of the task queue
    if(task_window_idx < self.task_window):
      # this is the non-masking version 
      for i in range(task_window_idx, self.task_window):
        self.first_task_pointers[i] = last_l_idx


  def step(self, action):
    reward = 0.0
    info = {}
    done = False
    self.current_iteration_time +=1 

    # make sure to remove masks as necessary
    # not used in current implementation
    # self.check_unmask_robot()

      
    # get the load and robot index from action
    # each robot-task combination corresponds
    # to one action in action space
    load_window_num = action % self.task_window
    l_idx = self.first_task_pointers[load_window_num]
    r_idx = int(action/self.task_window)
    # print("Load_ID ", l_idx, " assigned to Robot_ID ", r_idx)

    # ensure the load has not been assigned yet
    assert(self.state[l_idx][0] == 0)

    robot_loc = self.robots[r_idx][1]
    load_pickup = self.load_queue[l_idx][1]
    makespan, eie = self.robot_performance(r_idx, l_idx)

    # get the average relative load imbalance across robots
    avg_load_imbalance = 0
    for r_q in self.r_queue:
        relative_lb = (self.longest_queue_length-r_q)/self.longest_queue_length
        avg_load_imbalance += relative_lb 

    avg_load_imbalance = avg_load_imbalance/(self.num_robots*len(self.load_queue))
    # compute weighted reward
    # order of eie => 10^1, makespan => 10^2, load_imbalance=>10^-1
    reward = - (eie + 2*makespan/60 + 200*avg_load_imbalance)
    self.episode_reward_sum += reward
    
    # update state table and other metadata
    self.state[l_idx][0] = 1
    self.update_task_window_pointer()
    self.update_state_table()
    self.finished_loads += 1
    # print("Robot Timeframe: ", self.r_queue)
    
    if(self.finished_loads == (len(self.load_queue)-1)):
        self.episode_reward_mean = self.episode_reward_sum/len(self.load_queue)
        self.tot_episodes_rew_sum.append(self.episode_reward_sum)
        self.tot_episodes_rew_mean.append(self.episode_reward_mean)
        done = True

    return self.state, reward, done, info

  
  def mask_robot(self, r_idx):
    # mask any actions that select this robot
    for i in range(self.task_window):
      idx_ = (self.task_window*r_idx)+i
      self.legal_actions[idx_] = False

  def unmask_robot(self, r_idx):
    # unmask previously masked robot
    for i in range(self.task_window):
      idx_ = (self.task_window*r_idx)+i
      self.legal_actions[idx_] = True

  def check_unmask_robot(self):
    for r_i in range(self.num_robots):
      if(self.r_masked_iteration[r_i] == -1):
        continue
      else:
        diff = self.current_iteration_time - self.r_masked_iteration[r_i]
        if(diff >= self.charge_wait):
          self.r_masked_iteration[r_i] == -1
          self.r_battery[r_i] = self.robots[r_i][2]
          self.unmask_robot(r_i)


  def robot_performance(self, r_i, l_i):
  # r_idx => robot index
  # l_idx => load index
  # time_to => time robot took to reach the pickup node
  # returns the energy and completion time of a given load
    load_makespan = 0
    load_weight = self.load_queue[l_i][0]

    load_pickup = self.load_queue[l_i][1]
    load_dropoff = self.load_queue[l_i][2]
    
    load_travel_time = self.find_path(load_pickup, load_dropoff)
    total_travel_time = 2 * load_travel_time
    # consumed energy is calculated using load*time*battery_discharge_factor
    energy = (load_weight * total_travel_time) * self.robots[r_i][0]

    # update robot's battery charge level
    self.r_battery[r_i] -= energy
    
#     print("Battery Level for robot", r_i, ": ", round(self.r_battery[r_i],2))

    # make sure the robot has enough battery to carry load
    
    # if battery less than 15% do not assign task and add to queue the charge wait time
    if(self.r_battery[r_i] < 15):
      self.per_robot_charge_freq[r_i] += 1
      load_makespan += self.charge_wait
      self.r_queue[r_i] += self.charge_wait
      # reset battery to initial battery level
      self.r_battery[r_i] = self.robots[r_i][2]
      energy += 20
  

    # return robot's makespan (it includes charge time(if applicable))
    load_makespan += total_travel_time
    self.r_queue[r_i] += total_travel_time

    # checking the longest queue for load imbalance
    if (self.longest_queue_length < self.r_queue[r_i]):
        self.longest_queue_length = self.r_queue[r_i]

    # update the robot's location
    self.r_locs[r_i] = load_dropoff

    # add the load index to the robot's assigned loads
    self.assigned_loads_per_robot[r_i].append(l_i) 

    return load_makespan, energy

  def reset(self):
    self.episode_reward_sum = 0
    self.episode_reward_mean = 0
    self.r_queue = np.zeros(self.num_robots)
    self.r_masked_iteration = np.ones(self.num_robots)
    self.r_masked_iteration = -1*self.r_masked_iteration
    self.per_robot_charge_freq = np.zeros(self.num_robots)
    self.r_locs = []
    self.r_battery = []
    for bot in self.robots:
        self.r_locs.append(bot[1])
        self.r_battery.append(bot[2])

    self.assigned_loads_per_robot = np.empty((self.num_robots), dtype=object)
    for r in range(self.num_robots):
        self.assigned_loads_per_robot[r] = []

    self.finished_loads = 0
    self.first_task_pointers = np.zeros(self.task_window, dtype=int)
    for i in range(self.task_window):
        self.first_task_pointers[i] = i
    self.current_iteration_time = 0
    self.state = np.zeros((len(self.load_queue), ((2*self.num_robots)+1)), dtype=float)
    self.update_state_table()
    self.longest_queue_length = 0 
    self.legal_actions = np.ones(self.num_robots * self.task_window, dtype=bool)
    return self.state

  def state_representation(self):
    return {
        "action_mask" : self.legal_actions,
        "observation": self.state,
    }

  def render(self):
    # Do later
    pass 

### Experiment - Case 1

In [ ]:
# WAREHOUSE SETUP

num_robots = 5      # Number of robots
task_window = 4     # number of load feeder in docking area
charge_wait = 300    # Charging time
num_rack = 4        # Number of storage racks
units_per_rack = 5  # Number of units per rack
num_load = 250     # Number of loads to be sorted in the warehouse  
pickup_node = 1000  # Pickup node ID
dropoff_nodes = (num_rack*units_per_rack)*2 # Dropoff Node IDs
load_queue = create_load_dataset(num_load, pickup_node, dropoff_nodes)
robots  = create_robot_dataset(num_robots)

### Create GYM Environment

In [ ]:
env = MrtaEnv(load_queue, robots, num_robots, task_window, charge_wait, num_rack, units_per_rack )
env.reset()

### RL Policy + Model

In [ ]:
model = PPO("MlpPolicy", env, verbose=1)

### Model Learning

In [ ]:
model.learn(total_timesteps=10000000)

### Results

In [ ]:
x = np.arange(len(env.tot_episodes_rew_sum))
plt.plot(x, env.tot_episodes_rew_sum)
plt.xlabel("Training iterations")
plt.ylabel("Total reward")
plt.savefig("case1.png")

In [ ]:
import copy
def run_m(model):
  qs = []
  done = False
  obs = env2.reset()
  while not done:
    qs.append(copy.deepcopy(env2.r_queue))
    action, _states = model.predict(obs)
    obs, reward, done, info = env2.step(action)
  return qs

### Save Model

In [ ]:
model.save("ppo_mlp")

### Evaluate Result

In [ ]:
mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=300)
print("Mean Reward is: ", mean_reward)
print("Standard Reward is: ", std_reward)

## DAPSO Approach

### Robot Object
The class contains the methods and attributes that define a robot. 

The methods are as follows:
1. set_busy -> This sets the battery to busy
2. set_charging -> This sets the battery to charging
3. battery_history -> This stores the history of battery level
4. update_battery_history -> This updates the battery history
5. create_robot_dataset -> This creates a sample dataset of robots as per the given inputs
6. reset -> This resets the battery

In [ ]:
class Robot():
    def __init__(self, robot):
        self.bdf = robot['bdf']
        self.location = robot['location']
        self.battery_level = robot['init_battery']
        self.charge_time = robot['charge_time']
        self.available = True
        self.next_av_time = 0
        self.charging = False
        self.job = None
         # hard coded value
        self.battery_record = [self.battery_level]
        
    def set_busy(self, next_av_time, job_id):
        self.available = False
        self.charging = False
        self.next_av_time = round(next_av_time, 2)
        
    def set_charging(self, time):
        self.charging = True
        self.available = False
        self.next_av_time = round(time + self.charge_time, 2)
        self.battery_level = 100
        
    def battery_history(self):
        self.battery_record.append(self.battery_level)
        
    def update_battery_level(self, current_job_duration, job):
        energy = current_job_duration * job.weight * self.bdf
        self.battery_level -= round(energy, 2)
        self.battery_history()
        return energy
        
    def create_robot_dataset(num_bots, init_battery = 100):
        bots = []
        for j in range(num_bots): 
            # battery discharge factor is assumed as:
            # bdf = 100/(load*time_to_discharge)
            bdf = 100/(20*1000)
            charge_time = 300
            location = random.randint(1001,1002) #(0,j)
            bots.append(Robot({'bdf': bdf,'location': location,'init_battery': init_battery, 'charge_time':charge_time}))
        return bots
    
    def reset(self):
        self.battery_level = 100
        self.location = random.randint(1001,1002)
        self.battery_level = [self.battery_level]
        self.battery_record = [self.battery_level]
        self.available = True
        self.next_av_time = 0
        self.charging = False
        self.job = None
        
        return self

### Job Object

In [ ]:
class Job():
    def __init__(self, job):
        self.weight = job['weight']
        self.pickup_loc = job['pickup_loc']
        self.dropoff_loc = job['dropoff_loc']
        self.start_time = None
        self.end_time = None
        self.job_duration = None
        
    def update_start_time(self, time):
        self.start_time = time
        
    def update_end_time(self, time):
        self.end_time = round(time, 2)
        
    def update_job_duration(self, time):
        self.job_duration = round(time, 2)
        
    def create_load_dataset(num_loads, pickup_loc, grid_size):
        loads = []
        for i in range(num_loads): 
            #pick a random value between [5,25] for the load's weight
            weight = random.randint(5, 25)
            #compute the dropoff_locations
            dropoff_loc = random.randint(1,grid_size)
            val = {'weight': weight, 'pickup_loc': pickup_loc, 
                   'dropoff_loc': dropoff_loc}
            load = Job(val)
            loads.append(load)  
        return loads

### Schedule Class

In [ ]:
class Schedule():
    # schedule is particle
    def __init__(self, load_queue=None, robots=None, network=None):
        # array of tuple(job_id, bot_id)
        self.schedule = []
        self.load_queue = load_queue
        self.robots = robots
        self.av_job_ids = [i for i in range(len(load_queue))]
        self.completed_job_ids = []
        self.av_robot_ids = [i for i in range(len(robots))]
        self.nav_robot_ids = []
        self.ch_robot_ids = []
        self.completed_jobs = []
        self.fitness = None
        self.network = network
        self.cost = None
        self.pBest = None
        self.iterations = 0
        self.next_av_time_history = np.zeros(len(self.robots)) 
        self.battery_history = np.zeros(len(self.robots))
        self.time_record_df = pd.DataFrame(columns = self.record_df_columns())
        self.battery_record_df = pd.DataFrame(columns = self.record_df_columns())
        
        
    def find_path(self, source, destination):
        cost1 = self.network.best_route(source,destination)
        cost2 = self.network.best_route(destination, source)
        return cost1 + cost2

    def random_schedule(self):
        schedule = []
        for job_id, job in enumerate(self.load_queue):
            bot = random.choice(self.robots)
            bot_id = self.robots.index(bot)
            schedule.append((job_id, bot_id))
        return schedule
    
    
    def permute_schedule(self, prev_Schedule, best_Schedule, bias_param, inertia, duplicate_moves = 0):
        schedule = []
        count = 0
        i = 0
        while i < len(self.load_queue):
            if count < duplicate_moves and random.random() < bias_param:
                    schedule.append(best_Schedule[i])
                    count += 1      
            else:
                if random.random() < inertia:
                    schedule.append(prev_Schedule[i]) 
                else:
                    bot = random.choice(self.robots)
                    bot_id = self.robots.index(bot)
                    schedule.append((i, bot_id))
            i += 1
        
        return schedule
   
    def update_schedule(self, new_schedule, cost):
        self.schedule = new_schedule
        self.cost = cost
        
        
    def schedule_cost(self, given_schedule, store_data=False):
        time_steps = []
        i = 0
        energy = 0
        make_span = 0
        while(i < len(given_schedule)):
            (job_id, bot_id) = given_schedule[i]
            job = self.load_queue[job_id]
            bot = self.robots[bot_id]
            duration = self.find_path(job.pickup_loc, job.dropoff_loc)
            bot.set_busy(duration,job_id)
            # consumed energy is calculated using load*time*battery_discharge_factor
            energy += bot.update_battery_level(duration, job)
            
            # Update Makespan 
            make_span += duration
            
            #store_data records the cumulative work time of each robot
            if store_data:
                temp = np.zeros(len(self.robots)) 
                temp[bot_id] = duration
                self.next_av_time_history = np.add(self.next_av_time_history, temp)
                self.update_record(self.next_av_time_history)
                
            # add energy is bot is charging (assumes 80% charging efficiency)
            if bot.battery_level < 15:
                bot.set_charging(bot.next_av_time)
                energy += 20 
                make_span += bot.charge_time
                
                if store_data:
                    temp = np.zeros(len(self.robots)) 
                    temp[bot_id] = bot.charge_time
                    np.add(self.next_av_time_history, temp)
                    self.update_record(self.next_av_time_history)
                
            i += 1
        
        # load imbalance calculations
        time_list = [bot.next_av_time for bot in self.robots]
        max_time = max(time_list) 
        load_imbalance = 0

        for time in time_list:
            load_imbalance += (max_time - time)/max_time
            
        av_load_imbalance = load_imbalance/len(self.robots)
#         make_span = max_time   

        # order of energy => 10^1, makespan => 10^2, load_imbalance=>10^-1
        cost = (energy + 2*make_span/60 + 250*av_load_imbalance)
        self.update_schedule(given_schedule, cost)

        for bot in self.robots:
            bot = bot.reset()
            
        return make_span, cost
    
    def record_df_columns(self):
        column_name = []
        for i in range(len(self.robots)):
            column_name.append('Robot '+str(i))
        return column_name
            
    def update_record(self, data):
        self.time_record_df.loc[len(self.time_record_df)] = data.tolist()
        i = 0
        for bot in self.robots:
            self.battery_history[i] = np.array(bot.battery_level)
            i += 1
        self.battery_record_df.loc[len(self.battery_record_df)] = self.battery_history.tolist()
            
        
    def print_schedule(self):
        for k,v in self.schedule:
            print("Job: ", k, "Robot: ", v,
                  " start_time: ", self.load_queue[k].start_time, 
                  " end_time: ", self.load_queue[k].end_time,
                  " duration: ", self.load_queue[k].job_duration)

### PSO MRTA Solver

In [46]:
class TaskAllocSolverPSO():
    def __init__(self, scheduler, num_particles, iterations, 
                 cognitive_range, social_range, inertia, adaptive_control_inputs):
        self.scheduler = scheduler
        self.num_particles = num_particles
        self.iterations = iterations
        self.particles = []
        self.particle_prev_schedule = []
        self.best_schedule = []
        self.particle_best_cost = []
        self.particle_best_schedule = []
        self.gBest_cost = float('inf')
        self.gBest_schedule = []
        self.pBest_history = []
        self.cognitive_range = cognitive_range
        self.social_range = social_range
        self.inertia_range = inertia
        self.adaptive_control_inputs = adaptive_control_inputs
        self.cognitive_bias = 0
        self.social_bias = 0
        self.inertia_factor = inertia[1]
        self.copy_pBest_moves = 0
        self.copy_gBest_moves = 0
        
    def set_pBest(self):
        i = 0
        for particle in self.particles:
            time, pBest_candidate = particle.schedule_cost(particle.schedule)
            if(pBest_candidate < self.particle_best_cost[i]):
                self.particle_best_cost[i] = pBest_candidate
                self.particle_best_schedule[i] = particle.schedule
            self.pBest_history[i].append(self.particle_best_cost[i])
            i += 1
                
    def set_gBest(self):
        self.gBest_cost = min(self.particle_best_cost)
        index_min = min(range(len(self.particle_best_cost)), key=self.particle_best_cost.__getitem__)
        self.gBest_schedule = self.particle_best_schedule[index_min]
        
    def set_random_particles(self):
        for i in range(self.num_particles):
            schedule_ = self.scheduler.random_schedule()
            new_scheduler = Schedule(self.scheduler.load_queue, self.scheduler.robots, self.scheduler.network)
            time, fitness_value = new_scheduler.schedule_cost(schedule_)
            new_scheduler.update_schedule(schedule_, fitness_value)
            self.particles.append(new_scheduler)
            self.particle_prev_schedule.append(new_scheduler.schedule)
            self.particle_best_schedule.append(new_scheduler.schedule)
            self.particle_best_cost.append(fitness_value)
            self.pBest_history.append([])
            self.gBest_schedule = new_scheduler.schedule
        
    def update_particles(self):
        new_schedule = None
        for i in range(len(self.particles)):
            particle = self.particles[i]            
            new_schedule = self.scheduler.permute_schedule(self.particle_prev_schedule[i],
                                                           self.particle_best_schedule[i],
                                                           self.cognitive_bias,
                                                           self.inertia_factor,
                                                           self.copy_pBest_moves)

            new_schedule = self.scheduler.permute_schedule(new_schedule,
                                               self.gBest_schedule,
                                               self.social_bias,
                                               self.inertia_factor,
                                               self.copy_gBest_moves)
            if new_schedule != None:
                particle.schedule_cost(new_schedule)
                
    def run(self):
        pbar = tqdm(total=self.iterations)
        for i in range(self.iterations):
            self.adaptive_control(i)   
            pbar.update()
            self.update_particles()
            self.set_pBest()
            self.set_gBest()
    
    def adaptive_control(self, i):
        
        if i > self.iterations*self.adaptive_control_inputs[1]:
            self.inertia_factor = self.inertia_range[1]
            self.social_bias = self.social_range[1]
            self.cognitive_bias = self.cognitive_range[0]
            self.copy_pBest_moves = int(adaptive_control_inputs[2]*(i/self.iterations)*self.cognitive_bias)
            self.copy_gBest_moves = int(adaptive_control_inputs[2]*(i/self.iterations)*self.social_bias)
            return
            
        if i <= self.iterations*self.adaptive_control_inputs[0]:
            self.copy_pBest_moves = 0
            self.copy_gBest_moves = 0
            self.social_bias = self.social_range[0]
            self.cognitive_bias = self.cognitive_range[1]
            return
        
        if i <= self.iterations*self.adaptive_control_inputs[1]:
            self.social_bias = (self.social_range[0]+self.social_range[1])*0.5
            self.cognitive_bias = (self.cognitive_range[0]+self.cognitive_range[1])*0.5
            self.copy_pBest_moves = int(adaptive_control_inputs[2]*(i/self.iterations)*self.cognitive_bias)
            self.copy_gBest_moves = int(adaptive_control_inputs[2]*(i/self.iterations)*self.social_bias)
            return
            
    def visualize_pso(self, filename=None):
        fig, ax1 = plt.subplots()
        for i in range(0, len(self.particles)):
            ax1.plot(self.pBest_history[i])
               
        plt.xlabel("Iterations")
        plt.ylabel("pBest value")
        plt.show()
        if filename == None:
            filename = str(len(scheduler.load_queue)) + 'Loads_' + str(len(scheduler.robots)) + 'Bots'
        path = 'plots' + chr(92) + filename 
        fig.savefig(path +  ".eps", format='eps')
        fig.savefig(path + ".png")
        


### Experiment - Case 1

In [ ]:
# WAREHOUSE Setup

num_robots = 5      # Number of robots
num_rack = 4        # Number of storage racks
units_per_rack = 5  # Number of units per rack
num_load = 250      # Number of loads to be sorted in the warehouse  
pickup_node = 1000  # Pickup node ID

# Adaptive PSO setup
num_particles = 10
iterations = 2000
social_bias = [0.2, 0.8]
cognitive_bias = [0.1, 0.6]
inertia = [0.7, 0.9]

# adaptive_control_inputs requires three values:
# first parameter => range:[0-1], use: to switch social and cognitive bias parameter
# second parameter => range:[0-1], use: to switch social and cognitive bias parameter
# third parameter => range:[0-num_loads], use: dictates the max moves that can be duplicated
adaptive_control_inputs = [0.5, 0.9, num_load] #percentages

load_queue = Job.create_load_dataset(num_load, pickup_node, 2*num_rack*units_per_rack)
robots  = Robot.create_robot_dataset(num_robots)
network = LayoutNetworkX(num_rack, units_per_rack)
network.generate_graph()
network.best_route_store() #create lookup table for all possible routes

#initalize the scheduler
scheduler = Schedule(load_queue, robots, network)
# given_schedule = scheduler.random_schedule()
# make_span, reward = scheduler.schedule_cost(given_schedule)


solver = TaskAllocSolverPSO(scheduler,
                           num_particles,
                           iterations, 
                           cognitive_bias,
                           social_bias, 
                           inertia,
                           adaptive_control_inputs)

# initialize the particles with random schedules and run
solver.set_random_particles()
solver.run()

print("The Global Best Cost is: ", round(solver.gBest_cost,3))
print("The Global Best Schedule is: ")
print(solver.gBest_schedule)

### Solution Visualization

In [ ]:
solver.visualize_pso()

### Battery Level 

In [ ]:
scheduler2 = Schedule(load_queue, robots, network)
given_schedule = solver.gBest_schedule
make_span, reward = scheduler2.schedule_cost(given_schedule, True)
scheduler2.battery_record_df

### Time Record Details

In [ ]:
scheduler2.time_record_df

### Case 2

In [ ]:
# WAREHOUSE Setup

num_robots = 15      # Number of robots
num_rack = 6        # Number of storage racks
units_per_rack = 5  # Number of units per rack
num_load = 750      # Number of loads to be sorted in the warehouse  
pickup_node = 1000  # Pickup node ID

# Adaptive PSO setup
num_particles = 10
iterations = 2000
social_bias = [0.2, 0.8]
cognitive_bias = [0.1, 0.6]
inertia = [0.7, 0.9]

# adaptive_control_inputs requires three values:
# first parameter => range:[0-1], use: to switch social and cognitive bias parameter
# second parameter => range:[0-1], use: to switch social and cognitive bias parameter
# third parameter => range:[0-num_loads], use: dictates the max moves that can be duplicated
adaptive_control_inputs = [0.5, 0.9, num_load] #percentages

load_queue = Job.create_load_dataset(num_load, pickup_node, 2*num_rack*units_per_rack)
robots  = Robot.create_robot_dataset(num_robots)
network = LayoutNetworkX(num_rack, units_per_rack)
network.generate_graph()
network.best_route_store() #create lookup table for all possible routes

#initalize the scheduler
scheduler = Schedule(load_queue, robots, network)
# given_schedule = scheduler.random_schedule()
# make_span, reward = scheduler.schedule_cost(given_schedule)


solver_case_2 = TaskAllocSolverPSO(scheduler,
                           num_particles,
                           iterations, 
                           cognitive_bias,
                           social_bias, 
                           inertia,
                           adaptive_control_inputs)

# initialize the particles with random schedules and run
solver_case_2.set_random_particles()
solver_case_2.run()

print("The Global Best Cost is: ", round(solver.gBest_cost,3))
print("The Global Best Schedule is: ")
print(solver.gBest_schedule)

### Solution Visualization

In [ ]:
solver.visualize_pso()